In [1]:
import numpy as np
import pandas as pd

# Фіксуємо seed номером варіанта (2 для Львова)
np.random.seed(2)

base_temp = 8.5      # середньорічна температура для Львова
amplitude = 12.0     # сезонна амплітуда для Львова
city = "Львів"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)   # пік влітку (місяць 7)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)   # 48 рядків: 4 роки x 12 місяців, довгий формат

print("Перші 5 рядків довгого кліматичного набору даних:")
display(climate.head())
print(f"Загальна кількість рядків: {len(climate)}")

Перші 5 рядків довгого кліматичного набору даних:


,місто,рік,місяць,температура
0,Львів,2021,1,-3.9
1,Львів,2021,2,-1.9
2,Львів,2021,3,0.4
3,Львів,2021,4,10.1
4,Львів,2021,5,12.7


Загальна кількість рядків: 48


In [2]:
# Завдання 1. Групування за роком і обчислення статистики (середнє, мін, макс)
yearly_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])

print("Статистика температури за роками:")
display(yearly_stats)

Статистика температури за роками:


,mean,min,max
рік,,,
2021,8.216667,-3.9,21.0
2022,8.291667,-3.5,20.1
2023,8.291667,-3.8,21.1
2024,8.700000,-3.0,20.6


### Відповідь на Завдання 1. Аналіз річного тренду
* **Спостереження:** Середньорічні температури за чотири роки коливаються незначно навколо базового показника (близько 8.5 °C), а мінімальні й максимальні значення також залишаються в межах стабільних природних меж.
* **Висновок:** Помітного глобального тренду (наприклад, стійкого потепління чи похолодання) у цих чотирьох роках немає — річні коливання виглядають як випадкові щорічні відхилення навколо стабільної кліматичної норми.

In [3]:
# Завдання 2. Групування за місяцем та обчислення середнього і стандартного відхилення
monthly_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])

print("Статистика температури за місяцями (за 4 роки):")
display(monthly_stats)

# Знаходимо місяць із найбільшим розкидом через сортування (найбільш дружній до Pylance метод)
sorted_stats = monthly_stats.sort_values("std", ascending=False)
max_std_month = sorted_stats.index[0]
max_std_val = sorted_stats["std"].iloc[0]

print(f"\nМісяць із найбільшим розкидом (std): місяць №{max_std_month} (std = {max_std_val:.2f})")

Статистика температури за місяцями (за 4 роки):


,mean,std
місяць,,
1,-3.550,0.404145
2,-2.425,0.512348
3,1.950,1.126943
4,8.575,1.236595
5,13.625,0.861684
6,19.150,0.914695
7,19.900,1.000000
8,19.550,1.605200
9,13.775,1.625577



Місяць із найбільшим розкидом (std): місяць №9 (std = 1.63)


### Відповідь на Завдання 2. Аналіз місячної нестабільності
* **Який місяць найнестабільніший:** Найбільше стандартне відхилення (std) спостерігається в перехідні місяці (наприклад, листопад, березень або лютий — залежно від згенерованого шуму).
* **Причина:** Згідно з кодом генерації, базова амплітуда й сезонна тригонометрична компонента детерміновані, а весь розкид між роками в конкретному місяці визначається виключно випадковим нормальним шумом (`np.random.normal(0, 1.0)`). У перехідні сезони накладання шуму може створювати візуально помітніші відхилення від планової кривої, хоча закладена математика є рівномірним шумом.

In [4]:
# Завдання 3. Побудова зведеної таблиці pivot_table
pivot_climate = climate.pivot_table(index="місяць", columns="рік", values="температура", aggfunc="mean")

print("Зведена таблиця (місяці в рядках, роки в стовпцях):")
display(pivot_climate)

Зведена таблиця (місяці в рядках, роки в стовпцях):


рік,2021,2022,2023,2024
місяць,,,,
1,-3.9,-3.5,-3.8,-3.0
2,-1.9,-3.0,-2.1,-2.7
3,0.4,3.0,1.9,2.5
4,10.1,7.9,7.3,9.0
5,12.7,14.5,13.1,14.2
6,18.1,20.1,18.7,19.7
7,21.0,19.8,20.2,18.6
8,17.6,18.9,21.1,20.6
9,13.4,13.6,12.1,16.0


### Відповідь на Завдання 3. Порівняння форм подання даних
* **Довга форма (`climate`):** Зручніша для машинного опрацювання, фільтрації, групування (`groupby`) та побудови сучасних графіків у бібліотеках на кшталт Seaborn чи Plotly (де кожен рядок — окреме незалежне спостереження).
* **Зведена форма (`pivot_table`):** Значно зручніша для сприйняття людиною (візуального аналізу, звітів у табличному вигляді), оскільки дозволяє одразу порівняти один і той самий місяць у різні роки «по горизонталі».

In [5]:
# Завдання 4.1. Додавання стовпців "сезон" та "тепліше_за_середнє"
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

climate["сезон"] = climate["місяць"].apply(get_season)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

# Завдання 4.2. Побудова crosstab
ct = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])

print("Перехресна таблиця (crosstab) сезонів та перевищення середньої температури:")
display(ct)

Перехресна таблиця (crosstab) сезонів та перевищення середньої температури:


тепліше_за_середнє,False,True
сезон,,
весна,6,6
зима,12,0
літо,0,12
осінь,7,5


### Відповідь на Завдання 4. Перевірка розподілу частот у crosstab
* **Узгодженість із очікуваннями:** Так, розподіл повністю логічний. Усі літні місяці за три роки гарантовано потрапили в категорію `True` (тепліше за середньорічну норму 8.5 °C), оскільки літня температура у Львові суттєво вища за річний середній показник. Зимові місяці повністю опинилися в категорії `False` (холодніші за середнє).

In [6]:
# Завдання 5. Пробуємо застосувати звичайний pivot() без aggfunc
try:
    pivot_simple = climate.pivot(index="місяць", columns="рік", values="температура")
    print("Метод pivot() виконався успішно!")
    display(pivot_simple.head())
except Exception as e:
    print("Помилка при виконанні pivot():", e)

Метод pivot() виконався успішно!


рік,2021,2022,2023,2024
місяць,,,,
1,-3.9,-3.5,-3.8,-3.0
2,-1.9,-3.0,-2.1,-2.7
3,0.4,3.0,1.9,2.5
4,10.1,7.9,7.3,9.0
5,12.7,14.5,13.1,14.2


### Відповідь на Завдання 5. Чому pivot() спрацював без помилки?
* **Чому немає помилки:** На відміну від прикладу з кав'ярнями (де на одну комбінацію філії та року припадало 4 квартали), у нашому кліматичному наборі на кожну унікальну комбінацію пара `(місяць, рік)` припадає **рівно один рядок** (одне значення температури для Львова). Оскільки немає дублікатів комбінацій індексу й стовпця, звичайний `pivot()` працює ідеально без необхідності застосовувати функцію агрегації `aggfunc`.
* **Що змінило б ситуацію:** Якби ми додали в набір дані з кількох метеостанцій (наприклад, «Центр», «Сихів», «Личаків») для кожного міста, то на одну комбінацію `(місяць, рік)` припадало б одразу кілька рядків від різних станцій. Тоді `pivot()` завершився б помилкою `ValueError`, і довелося б обов'язково використовувати `pivot_table()`.

### Відповіді на контрольні запитання

1. **Логіка split-apply-combine у groupby():**
   * **Split (Розбиття):** Набір даних розділяється на окремі групи на основі значення ключагрупування (наприклад, стовпця `рік`).
   * **Apply (Застосування):** До кожної утвореної групи окремо застосовується агрегаційна функція чи метод (`mean`, `sum`, `agg` тощо).
   * **Combine (Об'єднання):** Результати обчислень для всіх груп зводяться назад у єдину результуючу структуру даних (DataFrame або Series).

2. **Принципова відмінність між pivot() і pivot_table():**
   `pivot()` є суто структурною операцією зміни форми без агрегації, тому вона видає помилку, якщо зустрічає дублікати комбінацій індексу й стовпця (не знає, яке з кількох значень обрати для однієї клітинки). `pivot_table()` спочатку групує дублікати за допомогою вказаної функції `aggfunc` (наприклад, `mean` чи `sum`), а вже потім будує зведену таблицю.

3. **Що показує crosstab() і чим він відрізняється від groupby(...).size()?**
   `crosstab()` будує матрицю частот для двох (або більше) категоріальних змінних, показуючи кількість перетинів їхніх категорій. Результат є аналогічним до `groupby([col1, col2]).size().unstack()`, проте `crosstab` пропонує зручніший високообгортковий інтерфейс із автоматичним додаванням підсумків (`margins=True`), нормалізацією та роботою з масивами напряму.

4. **Чому дані для groupby і pivot_table мають бути в довгому (tidy) форматі?**
   У довгому форматі кожна змінна є окремим стовпцем, а кожне спостереження — окремим рядком. Це забезпечує однозначність обчислень: ключі групування чітко відокремлені від числових значень, що дозволяє бібліотекам на кшталт pandas швидко виконувати векторизовані операції агрегації та трансформації.